# RunPod Vision OCR (standalone)

Single-purpose notebook: OCR scanned small-claims PDFs on a RunPod GPU using Qwen2-VL-7B 4-bit.

## How to use

1. **On your Mac** — produce a single-line base64 of your rclone config:
   ```
   base64 -i ~/.config/rclone/rclone.conf | pbcopy
   ```
2. **Deploy a RunPod pod** with an RTX 4090 / A5000 / similar (≥24 GB VRAM). `runpod/pytorch` template, 30+ GB container disk.
3. **In Jupyter Lab** on the pod: upload this `.ipynb` via the file browser, then open it.
4. Run cells **1–3** → paste the base64 string into Cell 3's hidden prompt.
5. Run Cell **4** (deps). When the banner says "Restart Kernel", do it. Then re-run Cells 1–3 (Cell 4 is skipped on the re-run since deps are already installed).
6. Run Cells **5 → 9** in order. Cell 8 is the long one (OCR loop, ~30s–1 min per scanned PDF on A5000).

## Notes

- No `git clone`, no `pip install -e`, no scraper-package imports. Self-contained.
- Uses `rclone copy` (not mount) — RunPod containers don't expose `/dev/fuse`.
- Safe to disconnect mid-OCR; the loop keeps running as long as the pod is alive.
- Whitelist of doc types is inlined from `scraper/config.py::DOC_TYPE_WHITELIST` — keep in sync if that list changes.

In [ ]:
import platform, subprocess
from pathlib import Path

print("Hostname:", platform.node())
print("OS:", platform.system(), platform.release())
print()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout.split("\n\n")[0])

# /workspace writability check (MooseFS occasionally goes RO mid-session — fail fast).
test_path = Path("/workspace/.write_test")
try:
    test_path.write_text("ok")
    test_path.unlink()
    print("/workspace: WRITABLE")
except Exception as e:
    raise RuntimeError(
        f"/workspace is not writable: {e}\n"
        "The MooseFS volume may be flaky. Terminate and redeploy the pod."
    )

In [ ]:
# Paste base64-encoded rclone.conf via a hidden prompt (no leak into notebook source).
# On Mac, copy it first:
#     base64 -i ~/.config/rclone/rclone.conf | pbcopy
# Then run this cell and paste at the prompt.
import base64, getpass, subprocess
from pathlib import Path

conf_path = Path.home() / ".config/rclone/rclone.conf"

if conf_path.exists():
    print(f"{conf_path} already exists — skipping upload.")
else:
    b64 = getpass.getpass("Paste base64 of rclone.conf (input hidden): ").strip()
    contents = base64.b64decode(b64).decode("utf-8")
    conf_path.parent.mkdir(parents=True, exist_ok=True)
    conf_path.write_text(contents)
    print(f"Wrote {conf_path} ({len(contents)} bytes)")

remotes = subprocess.run(["rclone", "listremotes"], capture_output=True, text=True).stdout.strip()
print(f"rclone remotes: {remotes!r}")
assert "drive:" in remotes, "Expected 'drive:' remote — check rclone.conf contents."

In [ ]:
# Install all runtime deps in one shot. cu124 torch works on driver 535+ (covers nearly all RunPod pods).
# AFTER THIS CELL: Kernel toolbar → Restart Kernel → re-run from Cell 1 (this cell will be skipped).
import subprocess, sys

print("Installing torch + torchvision (cu124) ...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "torch", "torchvision",
     "--index-url", "https://download.pytorch.org/whl/cu124"],
    check=True,
)
print("Installing transformers stack ...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers>=4.45", "accelerate", "bitsandbytes>=0.46.1",
     "qwen-vl-utils", "pymupdf", "Pillow"],
    check=True,
)

print("\n" + "=" * 60)
print("DEPS INSTALLED.")
print("NOW: Kernel toolbar → Restart Kernel, then re-run from Cell 1.")
print("=" * 60)

In [ ]:
# Sync Drive → local. Skips bulk copy if pdfs/ already has >100 files (saves a 1-2 min scan on re-runs).
# Set FORCE_RESYNC = True to pull new files from Drive after teammates upload.
FORCE_RESYNC = False

import subprocess
from pathlib import Path

DRIVE_DIR = Path("/workspace/litigation-pipeline")
PDF_DIR = DRIVE_DIR / "pdfs"
OUTPUT_DIR = DRIVE_DIR / "extracted"
PDF_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

n_local_pdfs = sum(1 for _ in PDF_DIR.iterdir())
n_local_txt = sum(1 for _ in OUTPUT_DIR.iterdir())

if n_local_pdfs > 100 and not FORCE_RESYNC:
    print(f"Local sync already populated: {n_local_pdfs:,} PDFs, {n_local_txt:,} .txt files.")
    print("Set FORCE_RESYNC = True at top of cell to pull new files from Drive.")
else:
    print("Syncing pdfs/ from Drive (one-time bulk download, ~10-30 min) ...")
    subprocess.run(
        ["rclone", "copy", "drive:litigation-pipeline/pdfs", str(PDF_DIR),
         "--transfers", "16", "--checkers", "32", "--progress"],
        check=True,
    )
    print("\nSyncing extracted/ from Drive ...")
    subprocess.run(
        ["rclone", "copy", "drive:litigation-pipeline/extracted", str(OUTPUT_DIR),
         "--transfers", "16", "--checkers", "32", "--progress"],
        check=True,
    )
    n_local_pdfs = sum(1 for _ in PDF_DIR.iterdir())
    n_local_txt = sum(1 for _ in OUTPUT_DIR.iterdir())
    print(f"\nDone — {n_local_pdfs:,} PDFs, {n_local_txt:,} .txt files.")

In [ ]:
# Find scanned PDFs that need OCR. Whitelist mirrors scraper/config.py::DOC_TYPE_WHITELIST.
from pathlib import Path

DOC_TYPE_WHITELIST = {
    "CLAIM_OF_PLAINTIFF",
    "DEFENDANT_S_CLAIM",
    "JUDGMENT",
    "ORDER",
    "DISMISSAL",
    "Notice_of_Entry_of_Judgment",
    "DECLARATION_OF_APPEARANCE",
    "STIPULATION",
    "COURT_JUDGMENT",
}


def is_doc_type_wanted(description: str) -> bool:
    desc_upper = description.upper()
    return any(w.upper() in desc_upper for w in DOC_TYPE_WHITELIST)


def _doc_type(p):
    return p.stem.split("_", 1)[-1] if "_" in p.stem else p.stem


all_pdfs = sorted(PDF_DIR.glob("*.pdf"))
whitelisted = [p for p in all_pdfs if is_doc_type_wanted(_doc_type(p))]
needs_gpu = [p for p in whitelisted if not (OUTPUT_DIR / f"{p.stem}.txt").exists()]

print(f"PDFs in pdfs/:             {len(all_pdfs)}")
print(f"Whitelisted doc types:     {len(whitelisted)}")
print(f"Already extracted to .txt: {len(whitelisted) - len(needs_gpu)}")
print(f"Need GPU vision model:     {len(needs_gpu)}")

In [ ]:
# Load Qwen2-VL-7B in bf16 with eager attention (flash_attn / sdpa producing token-loop
# garbage on some driver/torch combos). ~14 GB VRAM, fits on 24 GB GPUs.
model = None
processor = None

if not needs_gpu:
    print("Nothing to OCR — skipping model load.")
else:
    import torch
    from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available. Did you forget to restart the kernel after Cell 4?")

    MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"
    print(f"Loading {MODEL_ID} (bf16, eager attention) ...")
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        attn_implementation="eager",
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    used = torch.cuda.memory_allocated() / 1e9
    print(f"Loaded. GPU memory used: {used:.1f} GB")

In [ ]:
import io, time, subprocess
import fitz
from PIL import Image

GENERIC_PROMPT = (
    "This is a page from a San Francisco small claims court document. "
    "Extract all text exactly as it appears. Include party names, dates, "
    "claim amounts, rulings, and any other information on the page. "
    "Output plain text only."
)
DPI = 150

# Push extracted/ to Drive every SYNC_EVERY PDFs so a pod terminate costs at most
# SYNC_EVERY PDFs of work. Set to 0 to disable mid-run syncing.
SYNC_EVERY = 25


def page_to_pil(page, dpi=DPI):
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    return Image.open(io.BytesIO(pix.tobytes("png")))


def extract_page(pil_image, prompt):
    import torch
    from qwen_vl_utils import process_vision_info

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_image},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=2048, do_sample=False)
    generated = output_ids[0, inputs.input_ids.shape[1] :]
    return processor.decode(generated, skip_special_tokens=True)


def push_to_drive(quiet=True):
    """Incremental rclone copy of extracted/ → Drive. Safe to call mid-loop."""
    try:
        subprocess.run(
            ["rclone", "copy", str(OUTPUT_DIR), "drive:litigation-pipeline/extracted",
             "--transfers", "8", "--checkers", "16"],
            check=False,
            capture_output=quiet,
        )
        return True
    except Exception as e:
        print(f"  [sync] WARNING: rclone copy failed: {e}")
        return False


if needs_gpu and model is not None:
    import torch
    t0 = time.time()
    skipped_corrupt = []
    for i, pdf_path in enumerate(needs_gpu):
        txt_path = OUTPUT_DIR / f"{pdf_path.stem}.txt"
        if txt_path.exists():
            continue

        prompt = GENERIC_PROMPT
        try:
            doc = fitz.open(str(pdf_path))
        except Exception as e:
            print(f"  [{i + 1}/{len(needs_gpu)}] SKIP corrupt PDF: {pdf_path.name} ({e})")
            skipped_corrupt.append(pdf_path)
            continue

        pages_text = []
        for page_num, page in enumerate(doc, start=1):
            print(
                f"  [{i + 1}/{len(needs_gpu)}] {pdf_path.name} - page {page_num}/{len(doc)}",
                flush=True,
            )
            img = page_to_pil(page)
            text = extract_page(img, prompt)
            pages_text.append(text)
        doc.close()
        torch.cuda.empty_cache()

        full_text = "\n\n--- PAGE BREAK ---\n\n".join(pages_text)
        txt_path.write_text(full_text, encoding="utf-8")
        print(f"  Saved: {txt_path.name} ({len(full_text):,} chars)")

        if SYNC_EVERY and (i + 1) % SYNC_EVERY == 0:
            ok = push_to_drive()
            elapsed_min = (time.time() - t0) / 60
            print(f"  [sync {'ok' if ok else 'FAILED'}] {i + 1}/{len(needs_gpu)} done, {elapsed_min:.1f} min elapsed")

    # Final sync at end of run.
    print("\nFinal sync to Drive ...")
    push_to_drive(quiet=False)

    elapsed = time.time() - t0
    print(f"\nDone: {len(needs_gpu)} PDFs in {elapsed:.0f}s")
    if skipped_corrupt:
        print(f"\nSkipped {len(skipped_corrupt)} corrupt PDFs:")
        for p in skipped_corrupt:
            print(f"  {p.name}")
else:
    print("Nothing to do.")

In [ ]:
# Push new .txt files back to Drive. rclone copy is incremental — only new/changed files upload.
import subprocess

print("Syncing extracted/ → Drive ...")
subprocess.run(
    ["rclone", "copy", str(OUTPUT_DIR), "drive:litigation-pipeline/extracted",
     "--transfers", "16", "--checkers", "32", "--progress"],
    check=True,
)
n_local_txt = sum(1 for _ in OUTPUT_DIR.iterdir())
print(f"\nDone. {n_local_txt:,} .txt files now in Drive (and locally at {OUTPUT_DIR}).")